# EVA Symbolic — Обучение трансформера## FractalAttention: динамические головы, многомерные маски

In [ ]:
# 1. Клонирование
import os
if not os.path.exists('/home/jupyter/EVA'):
    !git clone https://github.com/BlackCatSpb/FCF.git /home/jupyter/EVA
%cd /home/jupyter/EVA
!git pull
!pip install loguru tokenizers numpy faiss-cpu datasets -q
import torch; print(f"CUDA: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 2. Загрузка affinity + создание трансформера
import sys, torch, numpy as np
sys.path.insert(0, '/home/jupyter/EVA')
from eva.symbolic import *
from eva.symbolic.fractal_attention import *
from eva.symbolic.advanced_methods import NGramContext
from eva.primordial_layer import PrimordialLayer
from eva.config import FCFConfig

pf = PotentialField(156, 256)
pf.load_state_dict(torch.load('checkpoints/symbolic/final/potential_field.pt', map_location='cpu', weights_only=True))
print(f"Affinity: mean={pf.affinity.mean():.4f}, std={pf.affinity.std():.4f}, max={pf.affinity.max():.4f}")

config = FCFConfig(); config.d_model=256; config.vocab_size=156; config.num_heads=8
layer = PrimordialLayer(config)
if device == 'cuda': layer = layer.cuda()
print(f"Transformer: {sum(p.numel() for p in layer.parameters()):,} params")

fractal_attn = FractalAttentionMask(d_model=256, num_base_heads=8)
if device == 'cuda': fractal_attn = fractal_attn.cuda()

char_vocab = CharacterVocab()

In [ ]:
# 3. Датасет: Wikipedia streaming + tokenization
from datasets import load_dataset
import re

print("Downloading Wikipedia RU...")
wiki = load_dataset("wikimedia/wikipedia", "20231101.ru", split="train", streaming=True)

all_ids = []; count = 0
for item in wiki:
    text = item.get("text", "")
    text = re.sub(r'[^а-яА-ЯёЁ\s.,;:!?\-—«»()""]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) < 20: continue
    ids = char_vocab.encode(text); ids.append(0)
    all_ids.extend(ids); count += 1
    if count % 50000 == 0:
        print(f"  {count:,} articles, {len(all_ids)/1e6:.1f}M tokens")
    if len(all_ids) > 150_000_000: break

all_ids = np.array(all_ids, dtype=np.int32)
print(f"Dataset: {len(all_ids)/1e6:.1f}M tokens from {count:,} articles")
!df -h /home/jupyter

In [ ]:
# 4. Обучение трансформера (knowledge distillation)
import torch.nn.functional as F, time, gc

BATCH, BLOCK = 128, 128
LR, STEPS = 1e-4, 30000

opt = torch.optim.AdamW(list(layer.parameters())+list(fractal_attn.parameters()), lr=LR)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
aff_tgt = pf.affinity.to(device)
V, PAD = pf.vocab_size, char_vocab.PAD_IDX

pos = 0; start = time.time(); loss_acc = 0
print(f"Training {STEPS} batches...")

for step in range(1, STEPS+1):
    if pos + BLOCK + 2 > len(all_ids): pos = 0
    ids_batch, lens = [], []
    for _ in range(BATCH):
        if pos + BLOCK + 2 > len(all_ids): pos = 0
        end = min(pos + BLOCK, len(all_ids))
        chunk = all_ids[pos:end]
        sep = np.where((chunk == 0) | (chunk == 3))[0]
        if len(sep) > 0 and sep[0] < BLOCK//2: end = pos + sep[0] + 1; chunk = all_ids[pos:end]
        ids = [int(x) for x in chunk if x >= 0][:BLOCK]
        ids_batch.append(ids); lens.append(len(ids)); pos += max(len(ids), 32)
    ml = max(lens)
    bt = torch.full((BATCH, ml), PAD, dtype=torch.long, device=device)
    for i, ids in enumerate(ids_batch): bt[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)

    layer.train(); fractal_attn.train()
    x = layer.embed(bt)
    hidden = layer.forward_transformer(x)
    logits = layer.forward_logits(hidden)

    tgt = aff_tgt[bt.clamp(0,V-1)]
    mask = (bt != PAD).float().unsqueeze(-1)
    log_p = F.log_softmax(logits, dim=-1)
    loss_logits = -(tgt * log_p * mask).sum() / (mask.sum() + 1e-8)

    attn = layer.transformer.attention.last_attention
    attn_avg = attn.mean(dim=1)[:, :ml, :ml]
    aff_t = aff_tgt[bt[:, :ml, None], bt[:, None, :ml]]
    loss_attn = F.mse_loss(attn_avg, aff_t.detach())

    try:
        fr_out = fractal_attn.multi_level_attention(x[:, :ml, :], bt[:, :ml])
        loss_fr = F.mse_loss(fr_out, hidden[:, :ml, :].detach())
    except: loss_fr = torch.tensor(0.0, device=device)

    loss = loss_logits * 0.4 + loss_attn * 0.3 + loss_fr * 0.3
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(layer.parameters(), 1.0)
    torch.nn.utils.clip_grad_norm_(fractal_attn.parameters(), 1.0)
    opt.step(); sch.step()
    loss_acc += loss.item()

    if step % 2000 == 0:
        elapsed = time.time() - start
        print(f"  step={step} | {step/max(elapsed,.01):.0f} b/s | loss={loss.item():.4f} | {elapsed/3600:.1f}h")
        gc.collect()
        if device == 'cuda': torch.cuda.empty_cache()

    if step % 10000 == 0:
        os.makedirs('/home/jupyter/EVA/checkpoints/transformer', exist_ok=True)
        torch.save(layer.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/weights.pt')
        torch.save(fractal_attn.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/fractal_attn.pt')
        print(f"  [Saved]")

os.makedirs('/home/jupyter/EVA/checkpoints/transformer', exist_ok=True)
torch.save(layer.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/weights_final.pt')
torch.save(fractal_attn.state_dict(), '/home/jupyter/EVA/checkpoints/transformer/fractal_attn_final.pt')
print(f"Done: {STEPS} steps in {time.time()-start:.0f}s")
!df -h /home/jupyter

In [ ]:
# 5. Тест генерации
layer.eval()
def gen(prompt, ml=80, t=0.6):
    ids = char_vocab.encode(prompt)[1:-1]; ctx = list(ids)
    for _ in range(ml):
        bt = torch.tensor([ctx[-128:]], dtype=torch.long, device=device)
        x = layer.embed(bt)
        with torch.no_grad():
            logits = layer.forward_logits(layer.forward_transformer(x))
            tp = F.softmax(logits[0,-1]/t, dim=-1)
            ap = F.softmax(torch.tensor(pf.get_continuation_potential(ctx[-1]).cpu().numpy(), device=device), dim=-1)
            cmb = 0.6 * tp + 0.4 * ap
        ns = torch.multinomial(cmb, 1).item()
        ctx.append(ns)
        if ns == char_vocab.EOS_IDX and len(ctx) > len(ids)+4: break
    return char_vocab.decode(ctx)

for p in ['pri', 'chelo', 'zem', 'pro']:
    print(f"  '{p}...' -> '{gen(p, 60)[:100]}'")

!cd /home/jupyter/EVA && tar czf /home/jupyter/transformer_trained.tar.gz checkpoints/transformer/
print(f"Packed: {os.path.getsize('/home/jupyter/transformer_trained.tar.gz')/1024/1024:.0f} MB")
print("Download: right-click -> Download")
!df -h /home/jupyter